# Aqua Blend – Barwon River Water Quality Cleaning

This notebook reconstructs the cleaning process used for the Aqua Blend water-quality CSV.

**Input:** `8d4703d2-74c1-4fb4-bcee-1b5568db34d9.csv`  
**Output:** `aquablend_water_quality_cleaned.csv`

The raw CSV contains three row layouts: normal rows, rows shifted left because the station name is missing, and rows shifted right because two fields are blank. The notebook repairs those rows, standardises dates/numeric fields, and adds a quality flag.

In [ ]:
import pandas as pd
import numpy as np

INPUT_FILE = '8d4703d2-74c1-4fb4-bcee-1b5568db34d9.csv'
OUTPUT_FILE = 'aquablend_water_quality_cleaned.csv'


## 1. Load the raw data

In [ ]:
df = pd.read_csv(INPUT_FILE, low_memory=False)

print('Rows and columns:', df.shape)
display(df.head())
print('\nMissing values:')
print(df.isna().sum())


## 2. Identify the three row structures

- **Normal rows:** fields are already in the expected positions.
- **Shifted-left rows:** the station name is missing, so the remaining fields have moved one column to the left.
- **Shifted-right rows:** `parameter` and `variable_code` are blank, while the real values appear later in the row.

In [ ]:
shifted_right = df['parameter'].isna() & df['variable_code'].isna()

shifted_left = (
    ~shifted_right
    & df['quality_code'].isna()
    & df['Unnamed: 9'].isna()
)

normal_rows = ~(shifted_right | shifted_left)

print('Normal rows:', int(normal_rows.sum()))
print('Shifted-left rows:', int(shifted_left.sum()))
print('Shifted-right rows:', int(shifted_right.sum()))


## 3. Rebuild the rows into the correct columns

In [ ]:
clean = pd.DataFrame(index=df.index)

# Start with the normal column positions
columns_to_copy = [
    'station',
    'station_name',
    'parameter',
    'variable_code',
    'datasource',
    'matched_variable_name',
    'datetime',
    'value',
    'quality_code'
]

for col in columns_to_copy:
    clean[col] = df[col]

# Repair shifted-left rows (station name missing)
clean.loc[shifted_left, 'station_name'] = np.nan
clean.loc[shifted_left, 'parameter'] = df.loc[shifted_left, 'station_name']
clean.loc[shifted_left, 'variable_code'] = df.loc[shifted_left, 'parameter']
clean.loc[shifted_left, 'datasource'] = df.loc[shifted_left, 'variable_code']
clean.loc[shifted_left, 'matched_variable_name'] = df.loc[shifted_left, 'datasource']
clean.loc[shifted_left, 'datetime'] = df.loc[shifted_left, 'matched_variable_name']
clean.loc[shifted_left, 'value'] = df.loc[shifted_left, 'datetime']
clean.loc[shifted_left, 'quality_code'] = df.loc[shifted_left, 'value']

# Repair shifted-right rows
clean.loc[shifted_right, 'parameter'] = df.loc[shifted_right, 'datasource']
clean.loc[shifted_right, 'variable_code'] = df.loc[shifted_right, 'matched_variable_name']
clean.loc[shifted_right, 'datasource'] = df.loc[shifted_right, 'datetime']
clean.loc[shifted_right, 'matched_variable_name'] = df.loc[shifted_right, 'value']
clean.loc[shifted_right, 'datetime'] = df.loc[shifted_right, 'quality_code']
clean.loc[shifted_right, 'value'] = df.loc[shifted_right, 'Unnamed: 9']
clean.loc[shifted_right, 'quality_code'] = np.nan

display(clean.head())


## 4. Standardise dates and numeric values

In [ ]:
clean['station'] = pd.to_numeric(clean['station'], errors='raise').astype('int64')
clean['variable_code'] = pd.to_numeric(clean['variable_code'], errors='raise').astype('int64')

clean['datetime'] = pd.to_datetime(
    clean['datetime'],
    errors='raise'
).dt.strftime('%Y-%m-%d')

clean['value'] = pd.to_numeric(clean['value'], errors='coerce')
clean['quality_code'] = pd.to_numeric(clean['quality_code'], errors='coerce')


## 5. Add a quality flag

- Quality codes **1** and **2** are kept as `available`.
- Quality code **255** is marked `check_quality`.
- Rows where no quality code was supplied are marked `quality_not_supplied`.

In [ ]:
clean['quality_flag'] = np.where(
    clean['quality_code'].isna(),
    'quality_not_supplied',
    np.where(
        clean['quality_code'].eq(255),
        'check_quality',
        'available'
    )
)

print(clean['quality_flag'].value_counts(dropna=False))


## 6. Final checks

In [ ]:
print('Final shape:', clean.shape)
print('\nMissing values after cleaning:')
print(clean.isna().sum())

print('\nDuplicate rows:', clean.duplicated().sum())
display(clean.head(10))


## 7. Save the cleaned CSV

In [ ]:
clean.to_csv(OUTPUT_FILE, index=False)
print(f'Cleaned file saved as: {OUTPUT_FILE}')
